# Validate Florence-2 + Groq Pipeline
This notebook tests the spacing corrections and token-overlap validations.

In [ ]:
import os
import subprocess

# Download the repository to get the samples/ folder and app.py
if not os.path.exists('samples'):
    print('Samples not found. Cloning repository...')
    subprocess.run(['git', 'clone', 'https://github.com/Aaryan658/iam-handwriting-explainer.git', 'repo'])
    os.rename('repo/samples', 'samples')
    os.rename('repo/app.py', 'app.py')
    print('Setup complete!')
else:
    print('Samples directory already exists.')

In [ ]:
!pip install -q transformers torch pillow groq wordninja jiwer datasets matplotlib

In [ ]:
import os
import jiwer
from PIL import Image
import app

In [ ]:
SAMPLES = [
    "samples/line_01.png",
    "samples/line_02.png",
    "samples/line_03.png",
    "samples/line_05.png",
    "samples/education_paragraph.png"
]

# Automatically inject Colab secrets into the environment variables
try:
    from google.colab import userdata
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
except Exception:
    if 'GROQ_API_KEY' not in os.environ:
        os.environ['GROQ_API_KEY'] = 'YOUR_GROQ_API_KEY_HERE'

for sample_path in SAMPLES:
    if not os.path.exists(sample_path):
        print(f"Sample not found: {sample_path}")
        continue
        
    print(f"\n--- Validating: {sample_path} ---")
    img = Image.open(sample_path).convert("RGB")
    txt_path = sample_path.replace(".png", ".txt")
    ground_truth = ""
    if os.path.exists(txt_path):
        with open(txt_path, "r", encoding="utf-8") as f:
            ground_truth = f.read().strip()
            
    print("Running Florence-2...")
    raw_ocr = app.transcribe(img)
    print(f"[RAW OCR]: {raw_ocr}")
    
    corrected_ocr = app.correct_spaces(raw_ocr)
    print(f"[SPACE CORRECTED]: {corrected_ocr}")
    
    if ground_truth:
        print(f"[GROUND TRUTH]: {ground_truth}")
        try:
            error = jiwer.wer(ground_truth.lower(), corrected_ocr.lower())
            accuracy = max(0.0, 1.0 - error)
            print(f"[WORD-LEVEL ACCURACY]: {accuracy:.2%}")
        except Exception as e:
            print(f"[WER CALCULATION ERROR]: {e}")
    else:
        print("[GROUND TRUTH]: None provided for this sample.")
        
    print("Running Groq explanation + Token overlap check...")
    explanation_result = app.explain(raw_ocr)
    print("\n[GROQ RESULT]:\n" + explanation_result)

## Performance Analysis
This section runs the TrOCR pipeline against a larger batch of 30 samples from the `Teklia/IAM-line` dataset to compute Word Error Rate (WER) and Character Error Rate (CER), and analyzes the distribution of Groq confidence scores.

In [ ]:
from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np

print("Loading 30 samples from Teklia/IAM-line dataset...")
# Use streaming to avoid downloading the entire large dataset
dataset = load_dataset("Teklia/IAM-line", split="train", streaming=True)

batch_samples = []
for i, item in enumerate(dataset):
    batch_samples.append({
        'image': item['image'].convert('RGB'),
        'text': item['text'].strip()
    })
    if len(batch_samples) >= 30:
        break

print(f"Successfully loaded {len(batch_samples)} samples.")

In [ ]:
wers = []
cers = []

confidence_counts = {"HIGH": 0, "MEDIUM": 0, "LOW": 0}
hallucination_overrides = 0
total_high_claims = 0

print("Running batch evaluation...")
for idx, sample in enumerate(batch_samples):
    ground_truth = sample['text']
    img = sample['image']
    
    # 1. Run TrOCR
    raw_ocr = app.transcribe(img)
    
    # Calculate Error Rates
    try:
        wer = jiwer.wer(ground_truth.lower(), raw_ocr.lower())
        cer = jiwer.cer(ground_truth.lower(), raw_ocr.lower())
        wers.append(wer)
        cers.append(cer)
    except:
        pass # Skip empty strings or jiwer failures
        
    # 2. Run Groq Explanation
    explanation = app.explain(raw_ocr)
    
    # 3. Parse Confidence and Overrides
    # We can detect an override if the output contains the specific override warning string
    is_overridden = "OVERRIDDEN from HIGH" in explanation
    
    if is_overridden:
        hallucination_overrides += 1
        total_high_claims += 1
        confidence_counts["MEDIUM"] += 1 # It gets downgraded to MEDIUM
    else:
        if "**Confidence:** HIGH" in explanation:
            confidence_counts["HIGH"] += 1
            total_high_claims += 1
        elif "**Confidence:** MEDIUM" in explanation:
            confidence_counts["MEDIUM"] += 1
        elif "**Confidence:** LOW" in explanation:
            confidence_counts["LOW"] += 1
            
print("Batch evaluation complete!")

In [ ]:
avg_wer = np.mean(wers) if wers else 0
avg_cer = np.mean(cers) if cers else 0
word_accuracy = max(0.0, 1.0 - avg_wer)
char_accuracy = max(0.0, 1.0 - avg_cer)

catch_rate = (hallucination_overrides / total_high_claims) if total_high_claims > 0 else 0

print("="*40)
print("SUMMARY STATISTICS (30 Samples)")
print("="*40)
print(f"Average Word Error Rate (WER): {avg_wer:.2%}")
print(f"Average Char Error Rate (CER): {avg_cer:.2%}")
print(f"Overall Word Accuracy:         {word_accuracy:.2%}")
print(f"Overall Char Accuracy:         {char_accuracy:.2%}")
print("-"*40)
print(f"Groq 'HIGH' Claims:            {total_high_claims}")
print(f"Hallucination Overrides:       {hallucination_overrides}")
print(f"Hallucination Catch Rate:      {catch_rate:.2%}")
print("="*40)

In [ ]:
# Create Visualizations
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Error Rates
metrics = ['WER', 'CER']
values = [avg_wer, avg_cer]
bars1 = ax1.bar(metrics, values, color=['#ef4444', '#f97316'])
ax1.set_title('Average Error Rates', fontsize=14, pad=15)
ax1.set_ylabel('Error Rate', fontsize=12)
ax1.set_ylim(0, max(max(values) * 1.2, 0.1) if values else 1)

# Add percentage labels on bars
for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
            f'{height:.1%}', ha='center', va='bottom', fontsize=11)

# Plot 2: Confidence Distribution
labels = ['HIGH', 'MEDIUM', 'LOW']
counts = [confidence_counts['HIGH'], confidence_counts['MEDIUM'], confidence_counts['LOW']]
bars2 = ax2.bar(labels, counts, color=['#22c55e', '#eab308', '#ef4444'])
ax2.set_title('Groq Final Confidence Distribution', fontsize=14, pad=15)
ax2.set_ylabel('Number of Samples', fontsize=12)
ax2.set_ylim(0, max(max(counts) * 1.2, 5) if counts else 10)

# Add count labels on bars
for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 0.2,
            f'{int(height)}', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()